# 50.5M Hindi–Sanskrit decoder

This is the largest size trained in experiment `010`. The trainable count is **50,476,544**. The command flag is `--model 50m`. The blueprint is `configs/model_50m.yaml`.

About a third of the weights are the 32,000-row embedding. The sibling notebooks are `model_10m.ipynb` and `model_18m.ipynb`.


## Architecture

Same block as the smaller decoders: grouped-query attention, SwiGLU, RMSNorm, rotary base 100,000, tied embeddings.

| Piece | Setting |
| --- | --- |
| Vocabulary | 32,000 |
| Width | 512 |
| Layers | 13 |
| Query heads | 8 |
| Key-value heads | 2 |
| Head dimension | 64 |
| SwiGLU intermediate | 1,280 |
| Context | 1,024 |
| Dropout | 0.05 |

| Weights | Count |
| --- | --- |
| Tied token embedding, 32,000 × 512 | 16,384,000 |
| One block: attention 655,360 + SwiGLU 1,966,080 + two RMSNorm scales 1,024 | 2,622,464 |
| Thirteen blocks | 34,092,032 |
| Final RMSNorm | 512 |
| **Total** | **50,476,544** |

Width equals 8 × 64. Output projections start at `0.02 / sqrt(2 * 13)`.


In [1]:
from pathlib import Path
import sys
import yaml

HERE = Path.cwd()
ROOT = HERE if (HERE / "configs" / "model_10m.yaml").exists() else HERE / "experiments" / "language" / "010-hindi-sanskrit-slm"
sys.path.insert(0, str(ROOT))
print("experiment", ROOT)

CONFIG_NAME = "configs/model_50m.yaml"
EXPECTED = 50_476_544

from src.model import build_model, count_parameters

config = yaml.safe_load((ROOT / CONFIG_NAME).read_text())
counted = count_parameters(config)
model = build_model(config)
live = model.num_parameters()
print("config", CONFIG_NAME)
print("formula", f"{counted:,}")
print("built", f"{live:,}")
print("expected", f"{EXPECTED:,}")
print("tied", model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr())
print(
    "layers", len(model.blocks),
    "width", config["n_embd"],
    "query heads", config["n_head"],
    "kv heads", config["n_kv_head"],
    "head dim", config["head_dim"],
    "context", config["block_size"],
    "vocab", config["vocab_size"],
)
assert live == EXPECTED == counted


experiment /Users/akashchauhan/Development/ai-models-learning/experiments/language/010-hindi-sanskrit-slm
config configs/model_50m.yaml
formula 50,476,544
built 50,476,544
expected 50,476,544
tied True
layers 13 width 512 query heads 8 kv heads 2 head dim 64 context 1024 vocab 32000


## Data and tokenizer

Tokenizer: `tokenizers/sp_32k.model`, 32,000 Unigram pieces, trained on the same cleaned pretrain text as the 10.9M run.

Pretraining repeated `data/clean/pretrain.jsonl` out to 500,009,664 tokens. That file is mostly Wikipedia, which later runs kept out. One fresh fine-tune then used `data/clean/sft.jsonl` for 20,000,534 supervised tokens, starting from `tok-500m`. The 10.9M checkpoint was not resized into this model. The vocabularies differ, so the runs do not resume each other.


## What was trained

Pretrain checkpoints were saved every 100M tokens from `tok-100m` through `tok-500m`. Pretrain loss moved from 4.99 at 100M tokens to 2.88 at 500M. The fine-tune loss ended at 3.32.

On the mixed internal validation split, chrF was **30.00** against a copy baseline of 25.63. The 10.9M fine-tune from `tok-200m` scored 26.57 on that same split. Cross-entropy is 3.72 here and 3.36 for the 10.9M model. Those two losses are not on the same vocabulary, so chrF is the comparison.

`panini-00-50m` is the Hugging Face name for this size. The checkpoint is not published yet, and it is not in git.


In [1]:
CHECKPOINTS = ["runs/sft/from-tok-500m"]

from src.sample import continue_prompt, load_checkpoint

PROMPTS = [
    ("translate_hi_sa", "पानी ठंडा है।"),
    ("translate_hi_sa", "मैं किताब पढ़ता हूँ।"),
    ("translate_sa_hi", "जलं शीतलम् अस्ति।"),
]

def show(directory, task, prompt, max_new_tokens=32):
    path = ROOT / directory
    if not (path / "checkpoint.pt").exists():
        print(f"missing {directory}")
        return None
    text, checkpoint = continue_prompt(
        path, task, prompt, max_new_tokens=max_new_tokens, temperature=0.0,
        model=show.model, tokenizer=show.tokenizer, device=show.device,
    )
    body = text.split("<tgt>", 1)[-1].strip() if "<tgt>" in text else text.strip()
    print(f"{task}")
    print(f"  prompt: {prompt}")
    print(f"  output: {body}")
    return checkpoint

for directory in CHECKPOINTS:
    path = ROOT / directory
    print("\n#", directory)
    if not (path / "checkpoint.pt").exists():
        print("checkpoint is not on this machine")
        continue
    show.model, show.tokenizer, checkpoint, show.device = load_checkpoint(path)
    print("stage", checkpoint["stage"], "tokens", f"{checkpoint['tokens_seen']:,}")
    for task, prompt in PROMPTS:
        show(directory, task, prompt)



# runs/sft/from-tok-500m
checkpoint is not on this machine


## Commands

```bash
uv run python -m src.smoke --model 50m
uv run python -m src.train --stage pretrain --model 50m --until-tokens 500000000 --tokenizer tokenizers/sp_32k.model --runs-dir runs/50m
uv run python -m src.train --stage sft --model 50m --init runs/50m/pretrain/tok-500m --tokenizer tokenizers/sp_32k.model --runs-dir runs/50m
uv run python -m src.sample --checkpoint runs/50m/sft/from-tok-500m --task translate_hi_sa --prompt "मैं किताब पढ़ता हूँ।"
```
